# **Question 9: Advanced - Testing & Mocking (Crucial for CI/CD)**

In MLOps, we build CI/CD pipelines. You cannot load a 5GB model or connect to a live Production Database every time you run your unit tests. It's slow and dangerous.

**The Scenario:**
You have a function `predict_user_churn(user_id)` that:
1.  Calls a database to get user features.
2.  Runs a model prediction.
3.  Returns the result.

You want to write a unit test for this function, but you **must not** actually connect to the database.

**The Question:**
1.  Which library in Python's standard library allows you to replace real objects with fake ones during testing?
2.  What is the specific command/decorator to replace the database connection function with a fake one?
3.  Why is "Dependency Injection" (passing the database client as an argument to the function rather than creating it inside) preferred for making this code testable?

### Part 1: The Testing Library

**Standard Library:** `unittest.mock` (built into Python 3.3+)

```python
from unittest.mock import Mock, patch, MagicMock
```

**Key components:**
- `Mock`: A fake object that records how it was called
- `MagicMock`: Mock with magic methods (`__str__`, `__len__`, etc.) pre-configured
- `patch`: Decorator/context manager to temporarily replace objects
- `mock_open`: Special mock for file operations

---

### Part 2: The Mocking Decorator

**The `@patch` decorator:**

```python
from unittest.mock import patch

# Original function (hard to test - has external dependencies)
def predict_user_churn(user_id):
    # Bad: Creates database connection internally
    conn = get_database_connection()
    user_data = conn.execute(f"SELECT * FROM users WHERE id = {user_id}")
    
    # Bad: Loads model internally
    model = load_model("churn_model.pkl")  # 5GB file!
    
    prediction = model.predict(user_data)
    return prediction

# Test with @patch decorator
@patch('my_module.load_model')           # Mock the model loading
@patch('my_module.get_database_connection')  # Mock the DB connection
def test_predict_user_churn(mock_db, mock_model):
    # Configure mock database
    mock_db.return_value.execute.return_value = {
        'age': 35, 
        'tenure': 24, 
        'monthly_spend': 150
    }
    
    # Configure mock model
    mock_model.return_value.predict.return_value = 0.73  # 73% churn risk
    
    # Call the function (uses mocks instead of real DB/model)
    result = predict_user_churn(user_id=123)
    
    # Assertions
    assert result == 0.73
    mock_db.return_value.execute.assert_called_once()
    mock_model.return_value.predict.assert_called_once()
```

**Key syntaxes:**

```python
# 1. As decorator (most common)
@patch('module.path.to.function')
def test_something(mock_function):
    mock_function.return_value = "fake result"
    # test code here

# 2. As context manager
def test_something():
    with patch('module.path.to.function') as mock_function:
        mock_function.return_value = "fake result"
        # test code here

# 3. Patching multiple targets (decorators stack bottom-to-top)
@patch('module.load_model')      # Third argument
@patch('module.get_database')    # Second argument  
@patch('module.call_api')        # First argument
def test_something(mock_api, mock_db, mock_model):
    # Decorators are applied bottom-to-top
    # Arguments are passed left-to-right
    pass
```

---

### Part 3: Dependency Injection for Testability

**The Problem (Tight Coupling):**

```python
# ❌ BAD: Hard-coded dependencies (not testable)
def predict_user_churn(user_id):
    # Creates dependencies INSIDE the function
    db = DatabaseConnection('prod-db.company.com')  # ← Hard to mock
    model = joblib.load('models/churn_v3.pkl')      # ← 5GB file load
    
    user_data = db.get_user_features(user_id)
    prediction = model.predict([user_data])
    
    return prediction[0]

# To test this, you MUST:
# - Have access to prod-db.company.com (dangerous!)
# - Load the 5GB model file (slow!)
# - Use @patch with complex import paths (brittle)
```

**The Solution (Dependency Injection):**

```python
# ✅ GOOD: Inject dependencies (easily testable)
def predict_user_churn(user_id, db_client, model):
    """
    Args:
        user_id: User identifier
        db_client: Database client (can be real or mock)
        model: ML model object (can be real or mock)
    """
    user_data = db_client.get_user_features(user_id)
    prediction = model.predict([user_data])
    return prediction[0]

# Production usage (real dependencies)
from database import DatabaseClient
from model_loader import load_churn_model

db = DatabaseClient('prod-db.company.com')
model = load_churn_model('models/churn_v3.pkl')
result = predict_user_churn(user_id=123, db_client=db, model=model)

# Test usage (fake dependencies - NO mocking needed!)
def test_predict_user_churn():
    # Create simple test doubles
    fake_db = Mock()
    fake_db.get_user_features.return_value = {'age': 35, 'tenure': 24}
    
    fake_model = Mock()
    fake_model.predict.return_value = [0.73]
    
    # Test is simple and clear
    result = predict_user_churn(
        user_id=123, 
        db_client=fake_db, 
        model=fake_model
    )
    
    assert result == 0.73
    fake_db.get_user_features.assert_called_with(123)
```

---

## 🧠 Core Concepts

### 1. Why We Mock in MLOps

**The Testing Pyramid for ML Systems:**

```
         ╱╲
        ╱  ╲         E2E Tests (Slow, Expensive)
       ╱ E2E╲        - Full pipeline with real data
      ╱──────╲       - Real model inference
     ╱        ╲      - Live database connections
    ╱Integration╲    
   ╱────────────╲   Integration Tests (Medium Speed)
  ╱              ╲  - Test component interactions
 ╱  Unit Tests    ╲ - Mock external dependencies
╱──────────────────╲
                     Unit Tests (Fast, Many)
                     - Pure logic testing
                     - Everything mocked
```

**Why mock?**

| Without Mocking | With Mocking |
|-----------------|--------------|
| Test takes 45 seconds | Test takes 0.05 seconds |
| Requires database access | No external dependencies |
| Needs 5GB model loaded | Lightweight fake object |
| Fails if DB is down | Always reliable |
| Can't test edge cases | Easy to simulate errors |
| Flaky (network issues) | Deterministic results |

---

### 2. Mock vs. Stub vs. Fake vs. Spy

**Test Double Taxonomy:**

```python
# 1. MOCK - Records interactions, asserts they happened correctly
mock_db = Mock()
mock_db.query.return_value = [{'user_id': 1}]
result = predict_churn(1, mock_db, model)
mock_db.query.assert_called_once_with(user_id=1)  # Verify behavior

# 2. STUB - Returns pre-programmed responses (no verification)
class StubDatabase:
    def query(self, user_id):
        return [{'user_id': user_id, 'age': 30}]  # Always returns this

stub_db = StubDatabase()
result = predict_churn(1, stub_db, model)  # Just returns data

# 3. FAKE - Working implementation, but simplified
class FakeDatabase:
    def __init__(self):
        self.data = {1: {'age': 30}, 2: {'age': 45}}  # In-memory dict
    
    def query(self, user_id):
        return self.data.get(user_id)

fake_db = FakeDatabase()
result = predict_churn(1, fake_db, model)

# 4. SPY - Real object that records calls
real_db = DatabaseClient()
spy_db = Mock(wraps=real_db)  # Wraps real object
result = predict_churn(1, spy_db, model)
spy_db.query.assert_called_once()  # Records real calls
```

**When to use each:**

| Type | Use Case | Example |
|------|----------|---------|
| **Mock** | Verify interactions | "Was API called with correct params?" |
| **Stub** | Return test data | "Simulate database returning users" |
| **Fake** | Complex logic | In-memory database, fake filesystem |
| **Spy** | Monitor real object | Debug production code behavior |

---

### 3. Dependency Injection Patterns

#### **Pattern 1: Constructor Injection**
```python
class ChurnPredictor:
    def __init__(self, db_client, model):
        self.db = db_client  # Inject via constructor
        self.model = model
    
    def predict(self, user_id):
        features = self.db.get_user_features(user_id)
        return self.model.predict([features])[0]

# Production
predictor = ChurnPredictor(
    db_client=ProductionDB(), 
    model=load_model('churn.pkl')
)

# Testing
predictor = ChurnPredictor(
    db_client=Mock(), 
    model=Mock()
)
```

#### **Pattern 2: Method Injection (Function Arguments)**
```python
def predict_churn(user_id, db_client, model):
    # Dependencies passed as arguments
    features = db_client.get_user_features(user_id)
    return model.predict([features])[0]

# Production
result = predict_churn(123, ProductionDB(), real_model)

# Testing
result = predict_churn(123, Mock(), Mock())
```

#### **Pattern 3: Property Injection**
```python
class ChurnPredictor:
    db_client = None  # Set externally
    model = None
    
    def predict(self, user_id):
        features = self.db_client.get_user_features(user_id)
        return self.model.predict([features])[0]

# Production
predictor = ChurnPredictor()
predictor.db_client = ProductionDB()
predictor.model = load_model('churn.pkl')

# Testing
predictor = ChurnPredictor()
predictor.db_client = Mock()
predictor.model = Mock()
```

**Which to use?**
- **Constructor injection**: Most common, explicit dependencies
- **Method injection**: Best for functions, functional programming
- **Property injection**: Use sparingly, less explicit

---

## 🔧 unittest vs. pytest

### Why pytest is Preferred in Modern MLOps

**Comparison:**

| Feature | `unittest` | `pytest` |
|---------|-----------|----------|
| **Syntax** | Verbose, class-based | Concise, function-based |
| **Assertions** | `self.assertEqual(a, b)` | `assert a == b` |
| **Fixtures** | `setUp`/`tearDown` methods | `@pytest.fixture` decorator |
| **Parameterization** | Manual loops | `@pytest.mark.parametrize` |
| **Output** | Basic | Rich, colored, detailed |
| **Plugins** | Limited | Extensive ecosystem |
| **Learning curve** | Steeper | Gentler |
| **Speed** | Slower | Faster (parallel execution) |

---

### unittest Example (Verbose)

```python
import unittest
from unittest.mock import patch, Mock

class TestChurnPrediction(unittest.TestCase):
    def setUp(self):
        """Runs before each test"""
        self.user_id = 123
        self.mock_db = Mock()
        self.mock_model = Mock()
    
    def tearDown(self):
        """Runs after each test"""
        pass
    
    @patch('my_module.load_model')
    @patch('my_module.get_database')
    def test_predict_high_churn(self, mock_get_db, mock_load_model):
        # Setup mocks
        mock_get_db.return_value = self.mock_db
        mock_load_model.return_value = self.mock_model
        
        self.mock_db.get_user_features.return_value = {'age': 25}
        self.mock_model.predict.return_value = [0.85]
        
        # Call function
        result = predict_user_churn(self.user_id)
        
        # Assertions
        self.assertEqual(result, 0.85)
        self.mock_db.get_user_features.assert_called_once_with(123)
    
    def test_predict_low_churn(self):
        self.mock_db.get_user_features.return_value = {'age': 55}
        self.mock_model.predict.return_value = [0.15]
        
        result = predict_user_churn(
            self.user_id, 
            self.mock_db, 
            self.mock_model
        )
        
        self.assertEqual(result, 0.15)

if __name__ == '__main__':
    unittest.main()
```

---

### pytest Example (Concise)

```python
import pytest
from unittest.mock import Mock

# Fixtures (setup/teardown, but reusable)
@pytest.fixture
def mock_db():
    """Provide a mock database client"""
    return Mock()

@pytest.fixture
def mock_model():
    """Provide a mock ML model"""
    return Mock()

# Test functions (no class needed!)
def test_predict_high_churn(mock_db, mock_model):
    # Setup
    mock_db.get_user_features.return_value = {'age': 25}
    mock_model.predict.return_value = [0.85]
    
    # Execute
    result = predict_user_churn(123, mock_db, mock_model)
    
    # Assert (simple assert statement!)
    assert result == 0.85
    mock_db.get_user_features.assert_called_once_with(123)

def test_predict_low_churn(mock_db, mock_model):
    mock_db.get_user_features.return_value = {'age': 55}
    mock_model.predict.return_value = [0.15]
    
    result = predict_user_churn(123, mock_db, mock_model)
    
    assert result == 0.15

# Parameterized tests (test multiple inputs easily)
@pytest.mark.parametrize("age,expected_churn", [
    (25, 0.85),  # Young user → high churn
    (35, 0.60),  # Middle-aged → medium churn
    (55, 0.15),  # Senior → low churn
])
def test_churn_by_age(mock_db, mock_model, age, expected_churn):
    mock_db.get_user_features.return_value = {'age': age}
    mock_model.predict.return_value = [expected_churn]
    
    result = predict_user_churn(123, mock_db, mock_model)
    assert result == expected_churn

# Run with: pytest test_churn.py -v
```

**Why pytest is better:**
- ✅ Less boilerplate (no class inheritance)
- ✅ Natural assertions (`assert` vs. `self.assertEqual`)
- ✅ Fixtures are reusable across tests
- ✅ Parametrization built-in
- ✅ Better error messages
- ✅ Parallel execution (`pytest -n auto`)

---

## 📊 Real MLOps Testing Scenarios

### Scenario 1: Testing Model Inference

**The Challenge:**
```python
# Production code
def batch_predict(user_ids: List[int]) -> List[float]:
    """Predict churn for multiple users"""
    # Problem 1: 5GB model loaded every time
    model = joblib.load('models/churn_v3.pkl')
    
    # Problem 2: Database connection in function
    db = DatabaseClient('prod-db.company.com')
    
    predictions = []
    for user_id in user_ids:
        features = db.get_user_features(user_id)
        pred = model.predict([features])[0]
        predictions.append(pred)
    
    return predictions
```

**The Solution (Dependency Injection + Mocking):**

```python
# Refactored for testability
def batch_predict(
    user_ids: List[int],
    db_client,  # Injected dependency
    model       # Injected dependency
) -> List[float]:
    """Predict churn for multiple users"""
    predictions = []
    for user_id in user_ids:
        features = db_client.get_user_features(user_id)
        pred = model.predict([features])[0]
        predictions.append(pred)
    
    return predictions

# Test (fast, no external dependencies)
def test_batch_predict():
    # Mock database
    mock_db = Mock()
    mock_db.get_user_features.side_effect = [
        {'age': 25, 'tenure': 6},   # user 1
        {'age': 45, 'tenure': 36},  # user 2
    ]
    
    # Mock model
    mock_model = Mock()
    mock_model.predict.side_effect = [
        [0.8],  # prediction for user 1
        [0.2],  # prediction for user 2
    ]
    
    # Execute
    result = batch_predict([1, 2], mock_db, mock_model)
    
    # Assert
    assert result == [0.8, 0.2]
    assert mock_db.get_user_features.call_count == 2
    assert mock_model.predict.call_count == 2
```

---

### Scenario 2: Testing Feature Extraction Pipeline

**The Challenge:**
```python
# Feature engineering that calls external APIs
def extract_features(user_id: int) -> Dict:
    # Problem: Calls multiple external services
    user_data = call_user_service_api(user_id)        # Slow API
    transaction_data = call_payment_api(user_id)      # Expensive API
    behavioral_data = query_clickstream_db(user_id)   # Large database
    
    features = {
        'age': user_data['age'],
        'total_spend': sum(t['amount'] for t in transaction_data),
        'page_views': len(behavioral_data),
    }
    return features
```

**The Solution (Mock All External Calls):**

```python
# Refactored
def extract_features(
    user_id: int,
    user_service,      # Inject API client
    payment_service,   # Inject API client  
    clickstream_db     # Inject database client
) -> Dict:
    user_data = user_service.get_user(user_id)
    transaction_data = payment_service.get_transactions(user_id)
    behavioral_data = clickstream_db.query_user_activity(user_id)
    
    features = {
        'age': user_data['age'],
        'total_spend': sum(t['amount'] for t in transaction_data),
        'page_views': len(behavioral_data),
    }
    return features

# Test (all services mocked)
def test_extract_features():
    # Mock services
    mock_user_service = Mock()
    mock_user_service.get_user.return_value = {'age': 30}
    
    mock_payment_service = Mock()
    mock_payment_service.get_transactions.return_value = [
        {'amount': 100}, {'amount': 200}
    ]
    
    mock_clickstream_db = Mock()
    mock_clickstream_db.query_user_activity.return_value = [
        {'page': '/home'}, {'page': '/products'}, {'page': '/checkout'}
    ]
    
    # Execute
    features = extract_features(
        user_id=123,
        user_service=mock_user_service,
        payment_service=mock_payment_service,
        clickstream_db=mock_clickstream_db
    )
    
    # Assert
    assert features == {
        'age': 30,
        'total_spend': 300,
        'page_views': 3
    }
```

---

### Scenario 3: Testing Model Training Pipeline (Advanced)

**The Challenge:**
```python
def train_churn_model(data_path: str) -> float:
    # Problem: Loads large dataset, trains model (slow!)
    df = pd.read_csv(data_path)  # 10GB file
    X = df.drop('churn', axis=1)
    y = df['churn']
    
    # Problem: Actual model training (takes hours)
    model = XGBClassifier()
    model.fit(X, y)
    
    # Problem: Saves to S3 (external dependency)
    save_to_s3(model, 'models/churn_v4.pkl')
    
    # Problem: Logs to MLflow (external dependency)
    mlflow.log_metric('accuracy', 0.87)
    
    return 0.87
```

**The Solution (Mock Everything Except Core Logic):**

```python
def train_churn_model(
    data_loader,     # Inject data loading
    model_trainer,   # Inject training logic
    model_saver,     # Inject model persistence
    metrics_logger   # Inject metrics logging
) -> float:
    # Load data (can be mocked with small dataset)
    df = data_loader.load_training_data()
    X = df.drop('churn', axis=1)
    y = df['churn']
    
    # Train model (can be mocked to skip training)
    model = model_trainer.train(X, y)
    
    # Save model (mocked to avoid S3 calls)
    model_saver.save(model, 'models/churn_v4.pkl')
    
    # Log metrics (mocked to avoid MLflow)
    accuracy = model.score(X, y)
    metrics_logger.log_metric('accuracy', accuracy)
    
    return accuracy

# Test (everything mocked, runs in milliseconds)
def test_train_churn_model():
    # Mock data loader (small fake dataset)
    mock_data_loader = Mock()
    mock_data_loader.load_training_data.return_value = pd.DataFrame({
        'age': [25, 35, 45],
        'tenure': [6, 24, 36],
        'churn': [1, 0, 0]
    })
    
    # Mock trainer (fake model that doesn't actually train)
    mock_trainer = Mock()
    fake_model = Mock()
    fake_model.score.return_value = 0.87
    mock_trainer.train.return_value = fake_model
    
    # Mock saver (doesn't actually save to S3)
    mock_saver = Mock()
    
    # Mock logger (doesn't actually log to MLflow)
    mock_logger = Mock()
    
    # Execute
    accuracy = train_churn_model(
        data_loader=mock_data_loader,
        model_trainer=mock_trainer,
        model_saver=mock_saver,
        metrics_logger=mock_logger
    )
    
    # Assert
    assert accuracy == 0.87
    mock_trainer.train.assert_called_once()
    mock_saver.save.assert_called_once()
    mock_logger.log_metric.assert_called_with('accuracy', 0.87)
```

---

## 🎯 Interview Talking Points

### Strong Statements to Make:

#### 1. **On Mocking:**
> "In MLOps CI/CD, we can't afford slow tests. Mocking allows us to isolate business logic from external dependencies—databases, APIs, file systems. Using `unittest.mock`, I can replace a 5GB model load with a lightweight `Mock` object that returns pre-configured predictions. This makes tests run in milliseconds instead of minutes, enabling fast feedback loops."

#### 2. **On unittest vs. pytest:**
> "While `unittest` is in the standard library and I'm comfortable with it, I prefer `pytest` for production ML systems. pytest's fixture system is more powerful than `setUp`/`tearDown`—fixtures are reusable across tests and composable. Plus, pytest's assertion introspection gives much better error messages, which saves debugging time. In my experience, pytest's ecosystem (pytest-cov for coverage, pytest-xdist for parallel execution) makes it the industry standard."

#### 3. **On Dependency Injection:**
> "Dependency injection is fundamental to testable code. Instead of creating dependencies inside functions—which couples your code to specific implementations—you pass them as arguments. This means in production I pass real `DatabaseClient` and `load_model()` results, but in tests I pass `Mock` objects. No complex patching, no brittle tests. The code explicitly declares its dependencies, which also improves readability."

#### 4. **On Test Speed:**
> "In MLOps pipelines, test speed directly impacts developer productivity. If tests take 10 minutes, developers won't run them locally before pushing. By mocking external dependencies—databases, S3, model loading—I can keep unit tests under 1 second total. We reserve slow integration tests for CI/CD, but developers get instant feedback from fast unit tests."

#### 5. **On Test Reliability:**
> "External dependencies make tests flaky. Network issues, database downtime, rate limits—all cause test failures unrelated to code changes. Mocking eliminates this. My unit tests are deterministic: same input always produces same output. If a test fails, it's a real bug, not a transient infrastructure issue."

---

## 📚 Advanced Mocking Techniques

### 1. Mocking Properties and Attributes

```python
# Mock object with attributes
mock_model = Mock()
mock_model.model_version = "v3.2.1"
mock_model.feature_names = ['age', 'tenure', 'spend']
mock_model.predict.return_value = [0.75]

assert mock_model.model_version == "v3.2.1"
```

### 2. Side Effects (Different Return Values)

```python
# Return different values on each call
mock_db = Mock()
mock_db.query.side_effect = [
    [{'user_id': 1}],  # First call
    [{'user_id': 2}],  # Second call
    [],                # Third call (no results)
]

assert mock_db.query() == [{'user_id': 1}]
assert mock_db.query() == [{'user_id': 2}]
assert mock_db.query() == []

# Simulate exceptions
mock_api = Mock()
mock_api.call.side_effect = TimeoutError("API timeout")

with pytest.raises(TimeoutError):
    mock_api.call()
```

### 3. Chained Method Calls

```python
# Mock chained calls: db.connection().cursor().execute()
mock_db = Mock()
mock_db.connection.return_value.cursor.return_value.execute.return_value = [
    {'id': 1, 'name': 'Alice'}
]

# Use it
result = mock_db.connection().cursor().execute("SELECT * FROM users")
assert result == [{'id': 1, 'name': 'Alice'}]
```

### 4. Asserting Call Arguments

```python
mock_api = Mock()
mock_api.send_request('https://api.example.com', method='POST', data={'key': 'value'})

# Check if called with specific arguments
mock_api.send_request.assert_called_with(
    'https://api.example.com',
    method='POST',
    data={'key': 'value'}
)

# Check call count
assert mock_api.send_request.call_count == 1

# Get call arguments for inspection
call_args = mock_api.send_request.call_args
assert call_args.args == ('https://api.example.com',)
assert call_args.kwargs == {'method': 'POST', 'data': {'key': 'value'}}
```

### 5. Partial Mocking (Spy Pattern)

```python
# Real object, but spy on method calls
real_model = XGBClassifier()
spy_model = Mock(wraps=real_model)

# Uses real implementation, but tracks calls
spy_model.fit(X_train, y_train)  # Actual training happens
spy_model.predict(X_test)        # Actual prediction happens

# But we can verify calls
spy_model.fit.assert_called_once()
spy_model.predict.assert_called_once()
```

### 6. Mocking Context Managers

```python
# Mock file operations
from unittest.mock import mock_open, patch

mock_file_data = "user_id,age,churn\n1,25,0\n2,35,1"

with patch('builtins.open', mock_open(read_data=mock_file_data)):
    with open('users.csv', 'r') as f:
        content = f.read()
    
assert content == mock_file_data

# Mock database connections (context manager)
mock_db = MagicMock()
mock_db.__enter__.return_value.cursor.return_value.fetchall.return_value = [
    {'id': 1}, {'id': 2}
]

with mock_db as conn:
    results = conn.cursor().fetchall()

assert results == [{'id': 1}, {'id': 2}]
```

---

## ⚠️ Common Pitfalls

### 1. **Patching the Wrong Import Path**

```python
# File: my_module.py
from database import DatabaseClient

def get_users():
    client = DatabaseClient()  # ← What are we patching?
    return client.query("SELECT * FROM users")

# ❌ WRONG: Patching the original module
@patch('database.DatabaseClient')
def test_get_users(mock_db):
    # This doesn't work! my_module imported DatabaseClient
    result = get_users()

# ✅ CORRECT: Patch where it's USED, not where it's DEFINED
@patch('my_module.DatabaseClient')  # Patch in my_module namespace
def test_get_users(mock_db):
    mock_db.return_value.query.return_value = [{'id': 1}]
    result = get_users()
    assert result == [{'id': 1}]
```

**Rule:** Patch where the object is **used**, not where it's **defined**.

---

### 2. **Forgetting to Configure Return Values**

```python
# ❌ WRONG: Mock returns Mock by default
mock_model = Mock()
result = mock_model.predict([[1, 2, 3]])
assert result == [0.75]  # FAILS! result is <Mock id='...'>

# ✅ CORRECT: Configure return value
mock_model = Mock()
mock_model.predict.return_value = [0.75]
result = mock_model.predict([[1, 2, 3]])
assert result == [0.75]  # PASSES
```

---

### 3. **Over-Mocking (Testing the Mocks, Not the Code)**

```python
# ❌ BAD: Too much mocking, no real logic tested
@patch('my_module.load_model')
@patch('my_module.preprocess_data')
@patch('my_module.extract_features')
@patch('my_module.validate_input')
def test_predict(mock_validate, mock_extract, mock_preprocess, mock_load):
    mock_validate.return_value = True
    mock_extract.return_value = {'age': 30}
    mock_preprocess.return_value = [1, 2, 3]
    mock_load.return_value.predict.return_value = [0.8]
    
    result = predict(user_id=123)
    assert result == 0.8
    
    # We're testing that mocks return what we told them to!
    # No actual business logic is tested.

# ✅ BETTER: Mock only external dependencies
def predict(user_id, model, db):  # Use DI
    # Real logic (tested):
    is_valid = validate_input(user_id)  # Real function
    if not is_valid:
        return None
    
    # Real logic (tested):
    features = extract_features(db.get_user(user_id))  # Real function
    
    # Mocked (external dependency):
    return model.predict([features])[0]

def test_predict():
    mock_model = Mock()
    mock_model.predict.return_value = [0.8]
    
    mock_db = Mock()
    mock_db.get_user.return_value = {'age': 30, 'tenure': 24}
    
    # Now we're testing real validate_input and extract_features logic
    result = predict(user_id=123, model=mock_model, db=mock_db)
    assert result == 0.8
```

**Rule:** Mock external dependencies, test your logic.

---

### 4. **Not Resetting Mocks Between Tests**

```python
# ❌ WRONG: Mock state persists
mock_db = Mock()

def test_first():
    mock_db.query()
    assert mock_db.query.call_count == 1

def test_second():
    mock_db.query()
    # FAILS! call_count is 2 (from previous test)
    assert mock_db.query.call_count == 1

# ✅ CORRECT: Create new mocks or use fixtures
@pytest.fixture
def mock_db():
    return Mock()  # Fresh mock for each test

def test_first(mock_db):
    mock_db.query()
    assert mock_db.query.call_count == 1

def test_second(mock_db):
    mock_db.query()
    assert mock_db.query.call_count == 1  # PASSES
```

---

### 5. **Mocking Too Early (Before Import)**

```python
# ❌ WRONG: Patching before module is imported
@patch('my_module.DATABASE_URL', 'fake-url')
import my_module  # ← Imports use real DATABASE_URL

# ✅ CORRECT: Import first, then patch
import my_module

@patch('my_module.DATABASE_URL', 'fake-url')
def test_something():
    # Now DATABASE_URL is patched
    pass
```

---

## 🔗 pytest Fixture Patterns

### Fixture Scopes (Performance Optimization)

```python
# Function scope (default) - runs for every test
@pytest.fixture
def mock_model():
    return Mock()

# Module scope - runs once per module
@pytest.fixture(scope="module")
def expensive_model():
    # Runs once for all tests in this file
    model = load_real_model()  # Expensive!
    return model

# Session scope - runs once per test session
@pytest.fixture(scope="session")
def database_connection():
    # Runs once for entire test suite
    conn = connect_to_test_db()
    yield conn
    conn.close()

# Class scope - runs once per test class
@pytest.fixture(scope="class")
def api_client():
    return APIClient(base_url="https://test.api.com")
```

---

### Fixture Composition

```python
# Base fixtures
@pytest.fixture
def mock_db():
    return Mock()

@pytest.fixture
def mock_model():
    model = Mock()
    model.predict.return_value = [0.75]
    return model

# Composed fixture (uses other fixtures)
@pytest.fixture
def churn_predictor(mock_db, mock_model):
    return ChurnPredictor(db=mock_db, model=mock_model)

# Test uses composed fixture
def test_prediction(churn_predictor):
    result = churn_predictor.predict(user_id=123)
    assert result == 0.75
```

---

### Parameterized Fixtures

```python
# Fixture that provides multiple values
@pytest.fixture(params=[0.1, 0.5, 0.9])
def churn_threshold(request):
    return request.param

# Test runs 3 times (once per parameter)
def test_churn_classification(churn_threshold):
    prediction = 0.6
    is_churner = prediction >= churn_threshold
    # Test runs with threshold = 0.1, 0.5, 0.9
```

---

## 🎓 Key Takeaways

### The Testing Hierarchy

```
1. Unit Tests (Fast, Isolated, Many)
   ↓ Mock all external dependencies
   ↓ Test pure business logic
   ↓ Run in CI on every commit
   
2. Integration Tests (Medium Speed, Moderate)
   ↓ Test component interactions
   ↓ Use test databases/services
   ↓ Run in CI before deployment
   
3. E2E Tests (Slow, Expensive, Few)
   ↓ Test full user workflows
   ↓ Use production-like environment
   ↓ Run nightly or pre-release
```

---

### Mental Model: What to Mock

```
ALWAYS MOCK:
✓ External APIs (slow, rate-limited, cost money)
✓ Databases (state, slow, requires setup)
✓ File I/O (especially large files like models)
✓ Network calls (unreliable, slow)
✓ Time-based operations (datetime.now(), sleep())
✓ Random number generation (non-deterministic)

NEVER MOCK:
✗ Your own business logic (defeats the purpose)
✗ Language built-ins (len, str, int) unless necessary
✗ Pure functions (no side effects, fast, deterministic)
✗ Simple data structures (dicts, lists)
```

---

### pytest Commands Cheat Sheet

```bash
# Run all tests
pytest

# Run with verbose output
pytest -v

# Run specific test file
pytest tests/test_churn.py

# Run specific test function
pytest tests/test_churn.py::test_predict_high_churn

# Run tests matching pattern
pytest -k "churn"

# Run with coverage report
pytest --cov=my_module --cov-report=html

# Run in parallel (faster!)
pytest -n auto

# Stop on first failure
pytest -x

# Show print statements
pytest -s

# Rerun failed tests
pytest --lf  # last failed

# Show slowest tests
pytest --durations=10
```

---

## 📖 Further Reading

**Essential concepts to explore:**
- Test-Driven Development (TDD): Write tests before code
- Behavior-Driven Development (BDD): `pytest-bdd` for Gherkin syntax
- Test coverage tools: `pytest-cov`, `coverage.py`
- Contract testing: `pact`, `vcr.py` for API mocking
- Property-based testing: `hypothesis` for generating test cases
- Mutation testing: `mutmut` to test your tests

---

## 🔑 Interview Cheat Sheet

| Question | Quick Answer |
|----------|--------------|
| What library for mocking? | `unittest.mock` (standard library) |
| Decorator to mock? | `@patch('module.path.to.function')` |
| Why dependency injection? | "Makes code testable by allowing fake dependencies in tests" |
| unittest vs pytest? | "pytest is preferred: less verbose, better fixtures, richer ecosystem" |
| What is a fixture? | "Setup code that runs before tests, provides test data/objects" |
| When to mock? | "Mock external dependencies: APIs, databases, file I/O, network" |
| When NOT to mock? | "Don't mock your own business logic—that's what you're testing" |

---

**One-Sentence Summary:**
> "In MLOps CI/CD, we use `unittest.mock` (or `pytest-mock`) to replace slow external dependencies—databases, models, APIs—with lightweight fake objects, enabling fast, reliable, deterministic unit tests that run in milliseconds and can be executed on every commit."

---

*Last updated: For MLOps/ML Engineer interview preparation*
*Focus: Testing, mocking, dependency injection, pytest best practices*